# Module 8C: DQ Dashboard - Streamlit in Snowflake

## Learning Objectives
- Build and deploy a **Streamlit in Snowflake (SiS)** DQ monitoring app
- Provide self-service DQ visibility to business users

> **Business Value:** Business users get a polished, interactive dashboard inside Snowflake. No Power BI license, no VPN, no external hosting -- just Snowflake credentials and a browser.

---
> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes | **Variant:** Streamlit in Snowflake
> **Reference:** [STUDENT_GUIDE.md](../guide/STUDENT_GUIDE.md) — Module 8 compares the 3 dashboard variants (Native, Python, Streamlit).


> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE DQ_LAB_WH;

---
## Shared Setup: Create DQ Reporting Views

> **Note:** These views are identical across all Module 8 variants (8A/8B/8C). If you already ran another variant, these views already exist -- running them again is safe (`CREATE OR REPLACE`).

> **Business Value:** BI tools cannot call table functions directly. Views provide a stable, queryable interface.

In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_RESULTS_FLAT AS
SELECT
    r.TABLE_NAME, r.METRIC_NAME,
    r.ARGUMENT_NAMES AS COLUMN_CHECKED, r.VALUE AS METRIC_VALUE,
    r.MEASUREMENT_TIME,
    COALESCE(c.SEVERITY, 'MEDIUM') AS SEVERITY,
    COALESCE(c.OWNER, 'Unassigned') AS RULE_OWNER,
    COALESCE(c.RULE_TYPE, 'SYSTEM') AS RULE_TYPE,
    CASE WHEN r.VALUE::NUMBER > 0 THEN 'FAIL'
         WHEN r.VALUE::NUMBER = 0 THEN 'PASS'
         ELSE 'NO_RESULT' END AS STATUS
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'
)) r
LEFT JOIN CORP_DWH.DQ.RULES_CATALOG c
    ON UPPER(r.METRIC_NAME) LIKE '%' || REPLACE(UPPER(c.RULE_NAME), ' ', '_') || '%';

> **What this does:** Creates V_DQ_SCORECARD view that calculates per-table health scores from the latest DQ expectation results.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_SCORECARD AS
WITH latest AS (
    SELECT TABLE_NAME, METRIC_NAME, COLUMN_CHECKED, STATUS, MEASUREMENT_TIME,
        ROW_NUMBER() OVER (PARTITION BY METRIC_NAME, COLUMN_CHECKED ORDER BY MEASUREMENT_TIME DESC) AS RN
    FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT
)
SELECT TABLE_NAME,
    COUNT(*) AS TOTAL_CHECKS,
    COUNT(CASE WHEN STATUS = 'PASS' THEN 1 END) AS PASSED,
    COUNT(CASE WHEN STATUS = 'FAIL' THEN 1 END) AS FAILED,
    ROUND(100.0 * COUNT(CASE WHEN STATUS = 'PASS' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS HEALTH_SCORE_PCT,
    MAX(MEASUREMENT_TIME) AS LAST_EVALUATED
FROM latest WHERE RN = 1 GROUP BY TABLE_NAME;

> **What this does:** Creates V_DQ_TREND view that provides hourly time-series data for charting quality metrics over time.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_TREND AS
SELECT DATE_TRUNC('HOUR', MEASUREMENT_TIME) AS MEASUREMENT_HOUR,
    METRIC_NAME, VALUE AS METRIC_VALUE,
    CASE WHEN VALUE > 0 THEN 'FAIL' ELSE 'PASS' END AS STATUS,
    MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
ORDER BY MEASUREMENT_TIME DESC;

> **What this does:** Creates V_DQ_EXECUTIVE_SUMMARY view that aggregates all checks into a single overall health percentage.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY AS
SELECT 'CORP_DWH' AS DATA_ESTATE, COUNT(*) AS TOTAL_CHECKS,
    COUNT(CASE WHEN STATUS = 'PASS' THEN 1 END) AS CHECKS_PASSING,
    COUNT(CASE WHEN STATUS = 'FAIL' THEN 1 END) AS CHECKS_FAILING,
    ROUND(100.0 * COUNT(CASE WHEN STATUS = 'PASS' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS OVERALL_HEALTH_PCT,
    CURRENT_TIMESTAMP() AS AS_OF
FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT;

---
## Create the App Schema and Stage

> **What this does:** Creates the APPS schema and a stage to host the Streamlit app files.


In [ ]:
CREATE SCHEMA IF NOT EXISTS CORP_DWH.APPS COMMENT = 'Streamlit applications';
CREATE OR REPLACE STAGE CORP_DWH.APPS.DQ_DASHBOARD_STAGE DIRECTORY = (ENABLE = TRUE);

---
## The Streamlit App Code

Below is the complete app. Review it, then we deploy it to Snowflake.

---
## App Features

The Streamlit app (`streamlit_dq_app.py`) provides 5 interactive tabs:

| Tab | Content |
|-----|--------|
| **Overview** | Pass/Fail charts, severity breakdown, top violations, health by table |
| **Trend** | Line charts showing quality over time (per hour + per metric) |
| **Failures** | Filterable table of all failing rules with severity filter |
| **Rules Catalog** | Provisioned vs pending rules, owner assignments |
| **Drill-Down** | Investigate specific issues: NULLs, duplicates, orphans, staleness |

> **Source code:** See `notebooks/streamlit_dq_app.py` in the repo for the full implementation.


---
## Deploy the App

> **What this does:** Uploads the Streamlit app code to the stage and creates the Streamlit object in Snowflake for deployment.


In [ ]:
from snowflake.snowpark.context import get_active_session
import tempfile, os
session = get_active_session()

# The full app code (same as streamlit_dq_app.py in the repo)
app_code = """import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

st.set_page_config(page_title="DQ Monitor", page_icon="\U0001F6E1", layout="wide")
session = get_active_session()

st.title("\U0001F6E1 Data Quality Monitoring Dashboard")
st.caption("Real-time quality metrics for CORP_DWH | Powered by Snowflake DMFs")

# --- KPI Header ---
exec_df = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY").to_pandas()

if not exec_df.empty:
    health = float(exec_df["OVERALL_HEALTH_PCT"].iloc[0] or 0)
    total = int(exec_df["TOTAL_CHECKS"].iloc[0] or 0)
    passing = int(exec_df["CHECKS_PASSING"].iloc[0] or 0)
    failing = int(exec_df["CHECKS_FAILING"].iloc[0] or 0)

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Overall Health", f"{health:.0f}%",
                delta="Healthy" if health >= 90 else "Degraded" if health >= 70 else "Critical",
                delta_color="normal" if health >= 90 else "off" if health >= 70 else "inverse")
    col2.metric("Total Checks", total)
    col3.metric("Passing", passing, delta=f"{passing}/{total}")
    col4.metric("Failing", failing, delta=f"-{failing}" if failing > 0 else "0",
                delta_color="inverse" if failing > 0 else "normal")
else:
    st.warning("No DQ results available yet. Run some DMF checks first (Modules 1-4).")
    st.stop()

st.divider()

# --- Tabs ---
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "\U0001F4CA Overview", "\U0001F4C8 Trend", "\U0001F6A8 Failures", "\U0001F4DD Rules Catalog", "\U0001F50D Drill-Down"
])

# === TAB 1: Overview (Charts) ===
with tab1:
    results_df = session.sql(
        "SELECT METRIC_NAME, COLUMN_CHECKED, METRIC_VALUE, STATUS, SEVERITY, RULE_TYPE "
        "FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT WHERE STATUS IN ('PASS', 'FAIL')"
    ).to_pandas()

    if results_df.empty:
        st.info("No results to visualize.")
    else:
        chart_col1, chart_col2 = st.columns(2)

        # Pass/Fail distribution (pie-like bar)
        with chart_col1:
            st.subheader("Pass/Fail Distribution")
            status_counts = results_df["STATUS"].value_counts().reset_index()
            status_counts.columns = ["Status", "Count"]
            st.bar_chart(status_counts.set_index("Status"), color=["#29B5E8"])

        # Failures by severity
        with chart_col2:
            st.subheader("Failures by Severity")
            fails_sev = results_df[results_df["STATUS"] == "FAIL"]["SEVERITY"].value_counts().reset_index()
            fails_sev.columns = ["Severity", "Count"]
            if not fails_sev.empty:
                st.bar_chart(fails_sev.set_index("Severity"), color=["#FF6B6B"])
            else:
                st.success("No failures!")

        # Top violations
        st.subheader("Top Violations (by value)")
        top_violations = (
            results_df[results_df["METRIC_VALUE"] > 0]
            .nlargest(10, "METRIC_VALUE")[["METRIC_NAME", "COLUMN_CHECKED", "METRIC_VALUE", "SEVERITY"]]
        )
        if not top_violations.empty:
            st.bar_chart(
                top_violations.set_index("METRIC_NAME")["METRIC_VALUE"],
                color="#FF9800"
            )
        else:
            st.info("All metrics at zero.")

        # Health by table (progress bars)
        st.subheader("Health by Table")
        scorecard = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_SCORECARD").to_pandas()
        if not scorecard.empty:
            for _, row in scorecard.iterrows():
                h = float(row["HEALTH_SCORE_PCT"] or 0)
                color = "normal" if h >= 90 else "off" if h >= 70 else "inverse"
                st.progress(h / 100, text=f"{row['TABLE_NAME']} — {h:.0f}% ({row['PASSED']}/{row['TOTAL_EXPECTATIONS']} passing)")
        else:
            st.info("No scorecard data.")

# === TAB 2: Trend (Line Chart) ===
with tab2:
    st.subheader("Quality Trend Over Time")
    trend_df = session.sql(\"\"\"
        SELECT MEASUREMENT_HOUR,
            COUNT(CASE WHEN STATUS = 'PASS' THEN 1 END) AS PASSING,
            COUNT(CASE WHEN STATUS = 'FAIL' THEN 1 END) AS FAILING
        FROM CORP_DWH.DQ.V_DQ_TREND
        GROUP BY MEASUREMENT_HOUR
        ORDER BY MEASUREMENT_HOUR
    \"\"\").to_pandas()

    if not trend_df.empty:
        trend_df = trend_df.set_index("MEASUREMENT_HOUR")
        st.line_chart(trend_df, color=["#4CAF50", "#f44336"])
        st.caption("Green = passing checks, Red = failing checks per measurement hour")
    else:
        st.info("Not enough historical data yet. Trend appears after multiple DMF evaluation cycles.")

    # Per-metric trend
    st.subheader("Individual Metric History")
    metric_trend = session.sql(\"\"\"
        SELECT MEASUREMENT_TIME, METRIC_NAME, METRIC_VALUE
        FROM CORP_DWH.DQ.V_DQ_TREND
        ORDER BY MEASUREMENT_TIME
    \"\"\").to_pandas()

    if not metric_trend.empty:
        metrics = sorted(metric_trend["METRIC_NAME"].unique())
        selected = st.multiselect("Select metrics to plot", metrics, default=metrics[:3])
        if selected:
            filtered = metric_trend[metric_trend["METRIC_NAME"].isin(selected)]
            pivot = filtered.pivot_table(index="MEASUREMENT_TIME", columns="METRIC_NAME", values="METRIC_VALUE")
            st.line_chart(pivot)
    else:
        st.info("No metric history available.")

# === TAB 3: Failures Detail ===
with tab3:
    st.subheader("Failing Rules")
    fails = session.sql(
        "SELECT METRIC_NAME AS RULE, COLUMN_CHECKED, METRIC_VALUE AS VIOLATIONS, "
        "SEVERITY, RULE_OWNER AS OWNER, MEASUREMENT_TIME AS LAST_CHECKED "
        "FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT WHERE STATUS = 'FAIL' AND METRIC_VALUE > 0 "
        "ORDER BY METRIC_VALUE DESC"
    ).to_pandas()

    if not fails.empty:
        # Severity filter
        severities = ["All"] + sorted(fails["SEVERITY"].unique().tolist())
        selected_sev = st.selectbox("Filter by severity", severities)
        if selected_sev != "All":
            fails = fails[fails["SEVERITY"] == selected_sev]

        st.dataframe(fails, use_container_width=True, hide_index=True)
        st.caption(f"Showing {len(fails)} failing rule(s)")
    else:
        st.success("\u2705 No failures detected! All expectations are passing.")

# === TAB 4: Rules Catalog ===
with tab4:
    st.subheader("Active Rules Catalog")
    catalog = session.sql(
        "SELECT RULE_NAME, RULE_TYPE, TARGET_TABLE, TARGET_COLUMN, SEVERITY, OWNER, "
        "CASE WHEN DMF_NAME IS NOT NULL THEN '\u2705 Provisioned' ELSE '\u23F3 Pending' END AS STATUS "
        "FROM CORP_DWH.DQ.RULES_CATALOG WHERE IS_ACTIVE = TRUE ORDER BY SEVERITY DESC"
    ).to_pandas()

    if not catalog.empty:
        # Summary metrics
        prov = len(catalog[catalog["STATUS"].str.contains("Provisioned")])
        pend = len(catalog) - prov
        mcol1, mcol2, mcol3 = st.columns(3)
        mcol1.metric("Total Rules", len(catalog))
        mcol2.metric("Provisioned", prov)
        mcol3.metric("Pending", pend)

        st.dataframe(catalog, use_container_width=True, hide_index=True)
    else:
        st.info("No rules in catalog. Run Module 3 first.")

# === TAB 5: Drill-Down ===
with tab5:
    st.subheader("Investigate Specific Issues")
    st.markdown(\"\"\"
    Use these queries to drill into specific DQ failures. Select a category below.
    \"\"\")

    drill = st.selectbox("Investigation type", [
        "NULL National IDs (CRM)",
        "Duplicate Customers",
        "Orphan Transactions",
        "Stale Data (Freshness)"
    ])

    if drill == "NULL National IDs (CRM)":
        df = session.sql(\"\"\"
            SELECT CUSTOMER_NAME, EMAIL, CITY, SOURCE_SYSTEM
            FROM CORP_DWH.SILVER.INT_CUSTOMERS
            WHERE NATIONAL_ID IS NULL
            ORDER BY CUSTOMER_NAME
        \"\"\").to_pandas()
        st.dataframe(df, use_container_width=True, hide_index=True)
        st.caption(f"{len(df)} customers with NULL National ID")

    elif drill == "Duplicate Customers":
        df = session.sql(\"\"\"
            SELECT CUSTOMER_NAME, NATIONAL_ID, SOURCE_SYSTEM, EMAIL, IS_DUPLICATE
            FROM CORP_DWH.SILVER.INT_CUSTOMERS
            WHERE IS_DUPLICATE = TRUE
            ORDER BY NATIONAL_ID
        \"\"\").to_pandas()
        st.dataframe(df, use_container_width=True, hide_index=True)
        st.caption(f"{len(df)} duplicate records flagged")

    elif drill == "Orphan Transactions":
        df = session.sql(\"\"\"
            SELECT TXN_ID, CUSTOMER_ID, CUSTOMER_REF, AMOUNT, TXN_TYPE, TXN_DATE
            FROM CORP_DWH.GOLD.FACT_TRANSACTIONS
            WHERE CUSTOMER_ID NOT IN (SELECT CUSTOMER_ID FROM CORP_DWH.GOLD.DIM_CUSTOMER)
            ORDER BY AMOUNT DESC LIMIT 20
        \"\"\").to_pandas()
        st.dataframe(df, use_container_width=True, hide_index=True)
        st.caption(f"{len(df)} orphan transactions (CUSTOMER_ID not in DIM_CUSTOMER)")

    elif drill == "Stale Data (Freshness)":
        df = session.sql(\"\"\"
            SELECT 'STG_TRANSACTIONS' AS TABLE_NAME,
                MAX(LOADED_AT) AS LAST_LOAD,
                DATEDIFF('HOUR', MAX(LOADED_AT), CURRENT_TIMESTAMP()) AS HOURS_SINCE_LOAD,
                CASE WHEN DATEDIFF('HOUR', MAX(LOADED_AT), CURRENT_TIMESTAMP()) > 2
                     THEN 'STALE' ELSE 'FRESH' END AS STATUS
            FROM CORP_DWH.RAW.STG_TRANSACTIONS
        \"\"\").to_pandas()
        st.dataframe(df, use_container_width=True, hide_index=True)
"""

# Write to a temp file with a clean name
tmp_dir = tempfile.mkdtemp()
app_path = os.path.join(tmp_dir, 'streamlit_dq_app.py')
with open(app_path, 'w') as f:
    f.write(app_code)

# Upload to stage
session.file.put(app_path, '@CORP_DWH.DQ.DQ_STAGE/streamlit/', auto_compress=False, overwrite=True)
print('App uploaded to @CORP_DWH.DQ.DQ_STAGE/streamlit/streamlit_dq_app.py')


> **Expected result:** Upload confirmed + app URL printed. Permission errors = check CORP_DQ_ADMIN role.

---
## Grant Access to Business Users

> **What this does:** Grants the DQ steward role access to the deployed Streamlit dashboard.


In [ ]:
GRANT USAGE ON STREAMLIT CORP_DWH.APPS.DQ_DASHBOARD TO ROLE CORP_DQ_STEWARD;

---
## Comparison: When to Use Each Variant

| Variant | Best For | Pros | Cons |
|---------|----------|------|------|
| **8A: Native Dashboards** | Quick SQL monitoring | Zero code, built-in | Limited interactivity |
| **8B: Python Charts** | Data engineers | Rich customization | Not shareable standalone |
| **8C: Streamlit (SiS)** | Business self-service | Interactive, deployed, shareable | Requires app code |
| **Power BI** | Enterprise BI teams | Familiar, RLS, scheduling | External tool, license |

---
## Checkpoint

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("=" * 50)
print("CHECKPOINT: Dashboard Views + Streamlit App")
print("=" * 50)

passed = 0
total = 5

# Check 4 dashboard views
for v in ['V_DQ_RESULTS_FLAT', 'V_DQ_SCORECARD', 'V_DQ_TREND', 'V_DQ_EXECUTIVE_SUMMARY']:
    try:
        cnt = session.sql(f"SELECT COUNT(*) AS C FROM CORP_DWH.DQ.{v}").collect()[0]['C']
        print(f"  [PASS] {v} -- {cnt} rows")
        passed += 1
    except Exception as e:
        print(f"  [FAIL] {v}: {str(e)[:50]}")

# Check Streamlit app exists
try:
    result = session.sql("SHOW STREAMLIT LIKE 'DQ_DASHBOARD' IN SCHEMA CORP_DWH.APPS").collect()
    if len(result) > 0:
        print(f"  [PASS] Streamlit app DQ_DASHBOARD deployed")
        passed += 1
    else:
        print(f"  [WAIT] Streamlit app not yet deployed (run the deploy cell above)")
except Exception as e:
    print(f"  [WAIT] Streamlit app not yet deployed: {str(e)[:50]}")

print(f"\nResult: {passed}/{total} checks passed")
print("=" * 50)


---
**Next:** Proceed to `9_TEARDOWN` (optional cleanup).